# Claim Type Classifier: Fine-tuning MuRIL with LoRA (English + Hinglish)
**Project:** Multimodal Motor Insurance Claim Agent (ShaikSure Motor Insurance, fictional)

**Goal:** read a claim description in **English or Hinglish** and predict the claim type:
`collision`, `theft`, `flood`, `fire`, `vandalism` or `glass_only`.

**Why MuRIL:** v1 used English DistilBERT and failed on Hinglish words like *gaadi* and *paani*.
MuRIL (Google) is a BERT model pre-trained on Indian languages, including Hindi written in English letters.

**Libraries:** pandas (tables), torch (runs the model), transformers (MuRIL + training), peft (LoRA),
scikit-learn (split + scores), openai (GPT comparison).

**Before running:** Runtime > Change runtime type > **T4 GPU**. `OPENAI_API_KEY` secret on (key icon). Then **Run all**.

In [ ]:
# Step 0: install the two libraries Colab does not have by default
!pip install -q peft openai
# Remove Colab's old torchao: peft refuses to load with it, and we don't use it
!pip uninstall -y -q torchao

In [ ]:
# Step 1: imports and settings
import json
import os
import shutil
import time

import pandas as pd
import torch
from peft import LoraConfig, TaskType, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, Trainer,
                          TrainingArguments, set_seed)

MODEL_NAME = "google/muril-base-cased"   # BERT model for Indian languages (Google)
MAX_LENGTH = 128                          # claim descriptions are short, 128 tokens is enough
EPOCHS = 10
SEED = 42                                 # same seed = same results every run
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

In [ ]:
# Step 2: load the data straight from the public GitHub repo
REPO = "https://raw.githubusercontent.com/IBRAHIM-1245/multimodal-motor-insurance-claim-agent/main"
data = pd.read_csv(f"{REPO}/data/claims/claim_descriptions.csv")
hard_en = pd.read_csv(f"{REPO}/data/claims/hard_test_claims.csv")
hard_hl = pd.read_csv(f"{REPO}/data/claims/hard_test_hinglish.csv")

print("Training data:", len(data), "| English hard set:", len(hard_en), "| Hinglish hard set:", len(hard_hl))
print(pd.crosstab(data["label"], data["language"]))

In [ ]:
# Step 3: turn label names into numbers, then split 80% train / 10% validation / 10% test
LABELS = ["collision", "theft", "flood", "fire", "vandalism", "glass_only"]
label2id = {name: i for i, name in enumerate(LABELS)}
id2label = {i: name for i, name in enumerate(LABELS)}

for df in (data, hard_en, hard_hl):
    df["label_id"] = df["label"].map(label2id)

# stratify = keep the same share of each claim type in every split
train_df, temp_df = train_test_split(data, test_size=0.2, stratify=data["label_id"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=SEED)
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test:", len(test_df))

In [ ]:
# Step 4: tokenize. The tokenizer splits text into sub-word tokens and turns them into ids.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ClaimDataset(torch.utils.data.Dataset):
    """Holds tokenized claim texts and their label ids, one item at a time for the Trainer."""

    def __init__(self, df):
        self.encodings = tokenizer(list(df["text"]), truncation=True, padding="max_length",
                                   max_length=MAX_LENGTH)
        self.labels = list(df["label_id"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: torch.tensor(values[i]) for key, values in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[i])
        return item


train_ds, val_ds, test_ds = ClaimDataset(train_df), ClaimDataset(val_df), ClaimDataset(test_df)
print(tokenizer.tokenize("Meri gaadi chori ho gayi"))

In [ ]:
# Step 5: load MuRIL with a 6-class head, then add LoRA
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                                  # rank: size of the small LoRA matrices
    lora_alpha=16,                        # scaling of the LoRA update
    lora_dropout=0.1,                     # dropout to reduce overfitting
    target_modules=["query", "value"],    # BERT's attention query and value layers
    modules_to_save=["classifier"],       # the new classification head is trained fully
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()       # shows how few parameters LoRA trains

In [ ]:
# Step 6: train. The Trainer runs the training loop and checks validation after every epoch.
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)        # highest score = predicted class
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}


args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=EPOCHS,
    learning_rate=5e-4,                   # LoRA uses a higher learning rate than full fine-tuning
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",                # check validation score after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,          # keep the epoch with the best validation F1
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    logging_steps=10,
    report_to="none",                     # no external logging service
    seed=SEED,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  compute_metrics=compute_metrics)

start = time.time()
trainer.train()
train_minutes = (time.time() - start) / 60
print(f"Training time: {train_minutes:.1f} minutes")

In [ ]:
# Step 7: a simple predict function (the FastAPI service will use the same logic)
model.eval()


def predict_claim_type(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]   # turn scores into probabilities (they add up to 1)
    best = int(torch.argmax(probs))
    return id2label[best], round(float(probs[best]), 3)


print(predict_claim_type("A lorry hit my car from behind at the signal and the bumper is broken"))
print(predict_claim_type("Kal raat meri gaadi ghar ke bahar se chori ho gayi"))

In [ ]:
# Step 8: evaluate on claims the model never saw during training
def evaluate(df, name):
    results = [predict_claim_type(text) for text in df["text"]]
    df["pred"] = [label for label, conf in results]
    df["confidence"] = [conf for label, conf in results]
    acc = accuracy_score(df["label"], df["pred"])
    f1 = f1_score(df["label"], df["pred"], labels=LABELS, average="macro", zero_division=0)
    print(f"\n===== {name}: accuracy {acc:.3f} | macro F1 {f1:.3f} =====")
    print(classification_report(df["label"], df["pred"], labels=LABELS, zero_division=0))
    return acc, f1


test_acc, test_f1 = evaluate(test_df, "Test set (same style as training)")
en_acc, en_f1 = evaluate(hard_en, "English hard set")
hl_acc, hl_f1 = evaluate(hard_hl, "Hinglish hard set")

# Confusion matrix on all hard claims: rows = true type, columns = predicted type
hard = pd.concat([hard_en, hard_hl], ignore_index=True)
confusion = pd.crosstab(hard["label"], hard["pred"], rownames=["true"], colnames=["predicted"])
print("\nConfusion matrix (all hard claims):")
print(confusion)

# Confidence: used on Day 3 to choose when to fall back to GPT
hard["correct"] = hard["label"] == hard["pred"]
print("\nAverage confidence when correct:", round(hard[hard["correct"]]["confidence"].mean(), 3))
print("Average confidence when wrong:  ", round(hard[~hard["correct"]]["confidence"].mean(), 3))

# Mistakes, for the model card
mistakes = hard[~hard["correct"]][["text", "language", "label", "pred", "confidence"]]
print(f"\nMistakes: {len(mistakes)} of {len(hard)}")
mistakes

In [ ]:
# Step 9: speed test, then save the LoRA adapter (small file) + tokenizer + label list
start = time.time()
for text in hard["text"]:
    predict_claim_type(text)
ft_ms = (time.time() - start) / len(hard) * 1000
print(f"Fine-tuned model: {ft_ms:.1f} ms per claim on {device}")

SAVE_DIR = "claim_classifier_lora"
if os.path.exists(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f"{SAVE_DIR}/labels.json", "w") as f:
    json.dump({"labels": LABELS, "base_model": MODEL_NAME, "max_length": MAX_LENGTH}, f, indent=2)

size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, n)) for n in os.listdir(SAVE_DIR)) / 1e6
print(f"Saved to {SAVE_DIR} ({size_mb:.1f} MB):", os.listdir(SAVE_DIR))

## Step 10: compare with zero-shot GPT-4o-mini
Uses the `OPENAI_API_KEY` secret. Never paste a key into a cell. Cost: 48 short calls, well under Rs 1.

In [ ]:
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

# gpt-4o-mini prices in USD per 1 million tokens. Check https://openai.com/api/pricing and update if changed.
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60

PROMPT = """You classify motor insurance claim descriptions written in English or Hinglish.
Reply with exactly one label from this list and nothing else:
collision, theft, flood, fire, vandalism, glass_only

Rules:
- Label the main cause of damage. An accident that also cracks the windshield is collision.
- Deliberate damage by people is vandalism, even if only glass broke.
- Burnt damage is fire, even if someone set it on purpose.
- glass_only only when nothing except glass is damaged.
- Stolen parts are theft."""


def gpt_classify(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[{"role": "system", "content": PROMPT}, {"role": "user", "content": text}],
    )
    answer = response.choices[0].message.content.strip().lower()
    cost = (response.usage.prompt_tokens * PRICE_INPUT + response.usage.completion_tokens * PRICE_OUTPUT) / 1e6
    return answer, cost


gpt_preds, gpt_cost = [], 0.0
start = time.time()
for text in hard["text"]:
    label, cost = gpt_classify(text)
    gpt_preds.append(label if label in LABELS else "invalid")   # invalid answers count as wrong
    gpt_cost += cost
gpt_ms = (time.time() - start) / len(hard) * 1000

hard["gpt_pred"] = gpt_preds
gpt_en_acc = accuracy_score(hard[hard["language"] == "en"]["label"], hard[hard["language"] == "en"]["gpt_pred"])
gpt_hl_acc = accuracy_score(hard[hard["language"] == "hinglish"]["label"], hard[hard["language"] == "hinglish"]["gpt_pred"])
gpt_acc = accuracy_score(hard["label"], hard["gpt_pred"])
print(f"GPT-4o-mini: all hard {gpt_acc:.3f} | English {gpt_en_acc:.3f} | Hinglish {gpt_hl_acc:.3f}")
print(f"Average time {gpt_ms:.0f} ms per claim | total cost ${gpt_cost:.5f}")

In [ ]:
# Step 11: side-by-side comparison on the same hard claims, then save results
ft_acc = accuracy_score(hard["label"], hard["pred"])
ft_en_acc = accuracy_score(hard[hard["language"] == "en"]["label"], hard[hard["language"] == "en"]["pred"])
ft_hl_acc = accuracy_score(hard[hard["language"] == "hinglish"]["label"], hard[hard["language"] == "hinglish"]["pred"])

comparison = pd.DataFrame([
    {"model": "MuRIL + LoRA (fine-tuned)", "english_acc": round(ft_en_acc, 3), "hinglish_acc": round(ft_hl_acc, 3),
     "all_hard_acc": round(ft_acc, 3), "ms_per_claim": round(ft_ms, 1), "usd_per_1000_claims": 0.0},
    {"model": "GPT-4o-mini (zero-shot)", "english_acc": round(gpt_en_acc, 3), "hinglish_acc": round(gpt_hl_acc, 3),
     "all_hard_acc": round(gpt_acc, 3), "ms_per_claim": round(gpt_ms, 1),
     "usd_per_1000_claims": round(gpt_cost / len(hard) * 1000, 4)},
])
print(comparison.to_string(index=False))

results = {
    "version": "v2",
    "base_model": MODEL_NAME,
    "languages": ["en", "hinglish"],
    "lora": {"r": 8, "alpha": 16, "dropout": 0.1, "target_modules": ["query", "value"]},
    "training": {"epochs": EPOCHS, "learning_rate": 5e-4, "train_rows": len(train_df), "val_rows": len(val_df),
                 "test_rows": len(test_df), "train_minutes": round(train_minutes, 2), "device": device},
    "test_set": {"accuracy": round(test_acc, 3), "macro_f1": round(test_f1, 3)},
    "english_hard_set": {"rows": len(hard_en), "accuracy": round(en_acc, 3), "macro_f1": round(en_f1, 3)},
    "hinglish_hard_set": {"rows": len(hard_hl), "accuracy": round(hl_acc, 3), "macro_f1": round(hl_f1, 3)},
    "fine_tuned_ms_per_claim": round(ft_ms, 1),
    "gpt_4o_mini": {"all_hard_accuracy": round(gpt_acc, 3), "english_accuracy": round(gpt_en_acc, 3),
                    "hinglish_accuracy": round(gpt_hl_acc, 3), "ms_per_claim": round(gpt_ms, 1),
                    "total_cost_usd": round(gpt_cost, 5)},
    "adapter_size_mb": round(size_mb, 2),
    "v1_baseline": {"base_model": "distilbert-base-uncased", "test_accuracy": 0.967, "english_hard_accuracy": 0.861,
                    "note": "English only; failed on Hinglish words (gaadi, paani) and unusual stolen parts"},
}
with open("classifier_results.json", "w") as f:
    json.dump(results, f, indent=2)
comparison.to_csv("model_comparison.csv", index=False)
confusion.to_csv("confusion_matrix.csv")
hard[["text", "language", "label", "pred", "confidence", "gpt_pred"]].to_csv("hard_test_predictions.csv", index=False)

In [ ]:
# Step 12: package everything in the same folders as the repo, then download one zip
if os.path.exists("day2_outputs"):
    shutil.rmtree("day2_outputs")
os.makedirs("day2_outputs/eval", exist_ok=True)
shutil.copytree(SAVE_DIR, f"day2_outputs/models/{SAVE_DIR}")
for name in ["classifier_results.json", "model_comparison.csv", "confusion_matrix.csv", "hard_test_predictions.csv"]:
    shutil.copy(name, f"day2_outputs/eval/{name}")
shutil.make_archive("day2_outputs_v2", "zip", "day2_outputs")

from google.colab import files
files.download("day2_outputs_v2.zip")